In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

# -----------------------------
# CONFIG
# -----------------------------
MODEL_NAME = "BAAI/bge-base-en-v1.5"
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA_DIR = "data"
OUT_DIR = "embeddings"
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# MODEL LOAD
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(DEVICE)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [3]:
# -----------------------------
# EMBEDDING FUNCTION
# -----------------------------
@torch.no_grad()
def encode_texts(texts):
    """
    Mean-pooling encoder.
    Returns numpy array of shape (N, 768)
    """
    all_embeddings = []

    for i in range(0, len(texts), BATCH_SIZE):
        batch = texts[i:i + BATCH_SIZE]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        ).to(DEVICE)

        outputs = model(**inputs)
        last_hidden = outputs.last_hidden_state  # (B, T, H)

        attention_mask = inputs["attention_mask"].unsqueeze(-1)
        masked_hidden = last_hidden * attention_mask
        summed = masked_hidden.sum(dim=1)
        counts = attention_mask.sum(dim=1)

        mean_pooled = summed / counts
        mean_pooled = torch.nn.functional.normalize(mean_pooled, p=2, dim=1)

        all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

# -----------------------------
# GENERIC CSV → PKL
# -----------------------------
def build_embedding_dict(df, id_col, text_col, out_path):
    ids = df[id_col].tolist()
    texts = df[text_col].astype(str).tolist()

    embeddings = encode_texts(texts)
    emb_dict = {id_: emb for id_, emb in zip(ids, embeddings)}

    with open(out_path, "wb") as f:
        pickle.dump(emb_dict, f)

    print(f"Saved {len(emb_dict)} embeddings → {out_path}")


DATA_DIR = r"C:\Users\admin\Desktop\ICML\datasets"
OUT_DIR = r"C:\Users\admin\Desktop\ICML\embeddings"


# -----------------------------
# DL-ID DATASET
# -----------------------------
dl_df = pd.read_csv(os.path.join(DATA_DIR, "dialogue.csv"))
build_embedding_dict(
    df=dl_df,
    id_col="dl_id",
    text_col="dialogue_text",
    out_path=os.path.join(OUT_DIR, "dialog_embeddings.pkl")
)

# -----------------------------
# R-ID DATASET
# -----------------------------
r_df = pd.read_csv(os.path.join(DATA_DIR, "response.csv"))
build_embedding_dict(
    df=r_df,
    id_col="r_id",
    text_col="response_text",
    out_path=os.path.join(OUT_DIR, "response_embeddings.pkl")
)

# -----------------------------
# D-ID DATASET
# -----------------------------
d_df = pd.read_csv(os.path.join(DATA_DIR, "documents.csv"))

# Option: concatenate headline + body (recommended)
d_df["full_text"] = d_df["headline"].astype(str) + " [SEP] " + d_df["newsbody"].astype(str)

build_embedding_dict(
    df=d_df,
    id_col="d_id",
    text_col="full_text",
    out_path=os.path.join(OUT_DIR, "doc_embeddings.pkl")
)

# -----------------------------
# S-ID DATASET
# -----------------------------
s_df = pd.read_csv(os.path.join(DATA_DIR, "summaries.csv"))
build_embedding_dict(
    df=s_df,
    id_col="s_id",
    text_col="summary",
    out_path=os.path.join(OUT_DIR, "summ_embeddings.pkl")
)

print("All embeddings generated successfully.")


Saved 25 embeddings → C:\Users\admin\Desktop\ICML\embeddings\dialog_embeddings.pkl
Saved 10 embeddings → C:\Users\admin\Desktop\ICML\embeddings\response_embeddings.pkl
Saved 30 embeddings → C:\Users\admin\Desktop\ICML\embeddings\doc_embeddings.pkl
Saved 10 embeddings → C:\Users\admin\Desktop\ICML\embeddings\summ_embeddings.pkl
All embeddings generated successfully.
